# Learning from Its Own Footprint — reproducible notebook**COMSCI/ECON 206 · Team FP3 · Feiyu Li, Xuantong Fu · Instructor: Prof. Luyao Zhang**> **When may an AI valuation rule revise itself from outcomes it helped cause?**This notebook reproduces every number quoted in the paper and on the A0 poster.Runtime: about 1 minute for the quick path, about 8 minutes for the full grid.**All data are synthetic. No claim is made about any real issuer.**

## 1. Get the code

In [ ]:
# On Colab, clone the repository. Locally, just run from the repo root.import os, sys, subprocessif not os.path.exists('src'):    subprocess.run(['git','clone','--depth','1',        'https://github.com/dku-comsci-econ206-Autumn2026/ps2-fp3.git','repo'], check=True)    os.chdir('repo')sys.path.insert(0, os.path.abspath('src'))print('cwd:', os.getcwd())print('files:', sorted(os.listdir('src')))

In [ ]:
!pip -q install 'numpy>=1.24' 'pandas>=2.0' 'matplotlib>=3.7'

## 2. Verify first, then trustThese 11 assertions are the paper's claims written as executable checks. They cover thetwo defects we found in our own PS2 code — the mis-normalised first-stage `F` and thepurge that read `b_true` — and the allocation fix the 14 October feedback asked for.

In [ ]:
!python tests/test_verification.py

## 3. The instructor's criticism, and the fix> *"Section 4 and `src/auction.py` currently allocate fixed shares"* — so changing a bid> never determined who received scarce capacity.Below, one bid is traced through quantity, price, proceeds and the resulting traininglabel, under both the new excludable rule and the retained PS2 rule.

In [ ]:
import numpy as npfrom model import Config, draw_issuer_typesfrom auction import investor_signals, posterior_bids, trace_one_bidcfg = Config()print(f'demand {cfg.total_demand:.0f} units vs {cfg.shares:.0f} shares '      f'-> coverage {cfg.coverage_ratio:.2f}x')rng = np.random.default_rng(7)theta, v, rho = draw_issuer_types(rng, cfg)x = investor_signals(theta, rng, cfg)bids = posterior_bids((1-cfg.beta)*cfg.mu0 + cfg.beta*cfg.b_true*theta, x, cfg, 0.4)marginal = int(np.argsort(-bids)[24])for rule in ('excludable', 'fixed_rationing'):    print(f'\n--- {rule} ---')    t = trace_one_bid(bids, cfg.with_(allocation_rule=rule), rng_seed=11,                      investor=marginal, delta=1.0)    for label, r in t.items():        y = theta + rho*r['proceeds_k']        print(f"  {label:9s} bid={r['bid']:6.3f}  qty={r['quantity']:5.2f}  "              f"P={r['clearing_price']:6.3f}  k={r['proceeds_k']:6.3f}  "              f"served={r['n_served']:2d}/50  training label y={y:7.3f}")

Under `excludable`, lowering the bid takes the quantity to **zero** and changes theclearing price, the proceeds and therefore the label the evaluator later learns from.Under the retained PS2 rule every investor is served regardless — which is exactly whythe PS2 allocation comparison could not do any work.

## 4. The headline grid

In [ ]:
!python src/run_experiment.py --quick     # 5 seeds, ~40 s; use the full run for the paper numbers

In [ ]:
import pandas as pdpd.set_option('display.width', 200, 'display.max_columns', 50)h = pd.read_csv('results/headline_table.csv')h[['mechanism','memory','mean_manipulation','price_rmse','feedback_bias',   'first_stage_F_pooled','retail_share','n_excluded','freeze_rate']].round(3)

## 5. The two self-challenges we set ourselves**Challenge 1.** *"The jump is from 200 synthetic issuers. If real IPO data doesn't showit, our story is a model artifact."*The placebo answers this inside the model: set `rho = 0` so capital cannot affectperformance, and the feedback channel is switched off at its source. If the effectsurvives, it is an artifact and we abandon the claim.

In [ ]:
import numpy as npfrom model import Configfrom simulation import rundef mean_m(cfg, mode, seeds=(42,43,44)):    return float(np.mean([run(s, mode, 'uniform_price', cfg)                          .query('cohort >= @cfg.burn_in_cohorts').m.mean() for s in seeds]))base = Config()for tag, cfg in (('baseline rho>0', base), ('PLACEBO rho=0', base.with_(rho_mean=0.0, rho_sd=0.0))):    nm, nh = mean_m(cfg, 'no_memory'), mean_m(cfg, 'naive_history')    print(f'{tag:16s} no_memory={nm:.3f}  naive={nh:.3f}  '          f'lift={nh-nm:+.3f} ({100*(nh-nm)/nm:+.1f}%)')

The lift collapses from roughly **+81%** to about **−5%**. Within the model the effect isthe capital channel, not the synthetic setup. External validity remains the roadmap'seconomics study.**Challenge 2.** *"Purging reduces bias but the first-stage F is only 0.136. If a strongerallocation shock still gives a low F, our correction doesn't work."*Two things turned out to be true. The `0.136` was partly a **defect** — the statistic wasdivided by the residual sum of squares instead of the residual variance. And thesubstantive worry was partly **right**: below the threshold the corrected estimator is notmerely biased, it is unstable.

In [ ]:
from evaluator import Evaluatorcfg = Config()df = run(42, 'capital_purged', 'uniform_price', cfg).query('cohort >= @cfg.burn_in_cohorts')ev = Evaluator('capital_purged', cfg)for r in df.itertuples():    ev.update(r.disclosure, r.performance, r.capital, r.psi)print(f'PS2 (legacy) formula : F = {ev.first_stage_F_legacy():8.4f}')print(f'textbook t^2         : F = {ev.first_stage_F():8.2f}')print(f'ratio                : 1 : {ev.first_stage_F()/ev.first_stage_F_legacy():.0f}')

In [ ]:
# How strong must the allocation shock be before the correction is trustworthy?# Averaged over 3 seeds. The paper's figure uses 10 -- a single path understates the# instability, because the blow-up below the threshold is erratic rather than systematic.from evaluator import _olsSEEDS = (42, 43, 44)def cell_stats(c, mode):    F, bias, frz = [], [], []    for s in SEEDS:        d = run(s, mode, 'uniform_price', c).query('cohort >= @c.burn_in_cohorts')        X = np.c_[np.ones(len(d)), d.disclosure, d.psi]        b, se, _ = _ols(X, d.capital.to_numpy())        F.append((b[2]/se[2])**2)        bias.append(d.b_hat.iloc[-1] - c.b_true)        frz.append(d.frozen.mean())    return np.mean(F), np.mean(bias), max(abs(x) for x in bias), np.mean(frz)rows = []for sd in (0.01, 0.02, 0.05, 0.08, 0.15, 0.25, 0.40):    c = cfg.with_(psi_sd_uniform=sd)    for mode in ('capital_purged', 'audited_freeze'):        F, bias, worst, frz = cell_stats(c, mode)        rows.append({'sigma_psi': sd, 'mode': mode, 'first_stage_F': F,                     'bias': bias, 'worst_abs_bias': worst, 'freeze_rate': frz})sweep = pd.DataFrame(rows)print(sweep.pivot(index='sigma_psi', columns='mode',                  values=['first_stage_F', 'worst_abs_bias']).round(3))print('\nWorst |bias| anywhere in the sweep:')for m in ('capital_purged', 'audited_freeze'):    print(f"  {m:16s} {sweep[sweep['mode']==m].worst_abs_bias.max():.3f}")print('\nThe paper quotes the 10-seed version: results/robust_instrument.csv')

Below `F = 10` the purged estimator is unreliable in a specific way: its error is not justlarger, it is **erratic** — across seeds it swings in both directions, which is thesignature of weak-instrument behaviour rather than of a constant bias. The audited ruledeclines to certify there, holds the committed parameter, and resumes automatically onceidentification arrives.Over the paper's full 10-seed sweep the purged estimator's worst case is **−2.563** whilethe audited rule's is **0.275**. A short 3-seed run understates the gap, so treat thetable above as an illustration and `results/robust_instrument.csv` as the evidence.**That gap is the paper's central claim.**

In [ ]:
!python src/robustness.py --quick

In [ ]:
!python src/make_figures.pyfrom IPython.display import Image, displayfor f in ('fig1_manipulation.png', 'fig2_instrument.png', 'fig3_placebo.png'):    display(Image(filename=f'figures/{f}'))

## 7. Frozen-run recordEvery run writes a record with the UTC timestamp, seeds, full parameter dump, libraryversions, platform and a SHA-256 prefix per source file — so a number in the paper canalways be traced back to the exact code that produced it.

In [ ]:
import jsonrec = json.load(open('results/fresh_run_record.json'))print('generated:', rec['generated_utc'])print('seeds    :', rec['seeds'])print('python   :', rec['python'], '| numpy', rec['numpy'], '| pandas', rec['pandas'])print('coverage :', rec['derived']['coverage_ratio'], 'x')print('\nsource hashes:')for k, v in rec['source_sha256_16'].items():    print(f'  {k:24s} {v}')

---**Licence:** MIT. **AI use:** AI-Assisted; disclosed in full in Appendix A.1 of the paper.Human authors remain responsible for all claims, sources and computation.